<a href="https://colab.research.google.com/github/vlaks524/DSCC-251-Final-Project/blob/main/notebooks/ver2_tfidf_LR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report

In [ ]:
#Loading FinancialPhraseBank file (75 Agree)
FILE_PATH = "/content/Sentences_75Agree.txt"

# read raw lines
with open(FILE_PATH, "r", encoding="utf-8", errors="replace") as f:
    lines = f.readlines()

# parse "sentence@label"
rows = []
for line in lines:
    line = line.strip()
    if not line:
        continue

    # split from the right just in case "@" appears in text
    parts = line.rsplit("@", 1)
    if len(parts) != 2:
        continue

    text, label = parts
    text = text.strip()
    label = label.strip().lower()

    if text and label in ["positive", "neutral", "negative"]:
        rows.append((text, label))

df = pd.DataFrame(rows, columns=["text", "label"])

print("Shape:", df.shape)
print(df.head())
print("\nLabel counts:")
print(df["label"].value_counts())

In [ ]:
#Encoding labels
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

print("Classes:", list(label_encoder.classes_))

In [ ]:
#Splitting into pool and test
X = df["text"].tolist()
y = df["label_id"].tolist()

X_pool, X_test, y_pool, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Pool size:", len(X_pool))
print("Test size:", len(X_test))

In [ ]:
#Creating initial labeled set + unlabeled pool
INITIAL_LABEL_SIZE = 60

pool_indices = np.arange(len(X_pool))

initial_indices, unlabeled_indices = train_test_split(
    pool_indices,
    train_size=INITIAL_LABEL_SIZE,
    random_state=42,
    stratify=np.array(y_pool)
)

print("Initial labeled size:", len(initial_indices))
print("Initial unlabeled size:", len(unlabeled_indices))

In [ ]:
#TF-IDF + Logistic Regression
def make_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words="english"
        )),
        ("clf", LogisticRegression(
            max_iter=2000,
            multi_class="auto"
        ))
    ])

In [ ]:
#Evaluation helper
def evaluate_model(model, X_test, y_test):
    preds = model.predict(X_test)

    return {
        "accuracy": accuracy_score(y_test, preds),
        "f1_macro": f1_score(y_test, preds, average="macro"),
        "preds": preds
    }

In [ ]:
#Training model on initial 60 samples
X_labeled = [X_pool[i] for i in initial_indices]
y_labeled = [y_pool[i] for i in initial_indices]

model = make_model()
model.fit(X_labeled, y_labeled)

results = evaluate_model(model, X_test, y_test)

print("Initial results on 60 labeled samples")
print("Accuracy:", results["accuracy"])
print("Macro F1:", results["f1_macro"])

print("\nClassification report:\n")
print(classification_report(
    y_test,
    results["preds"],
    target_names=label_encoder.classes_,
    zero_division=0
))

In [ ]:
#Random sampling loop
def run_random_sampling(
    X_pool, y_pool, X_test, y_test,
    initial_size=60,
    batch_size=50,
    rounds=8,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    X_pool = np.array(X_pool, dtype=object)
    y_pool = np.array(y_pool)

    all_indices = np.arange(len(X_pool))

    labeled_indices = rng.choice(all_indices, size=initial_size, replace=False)
    unlabeled_indices = np.setdiff1d(all_indices, labeled_indices)

    history = []

    for r in range(rounds):
        X_labeled = X_pool[labeled_indices].tolist()
        y_labeled = y_pool[labeled_indices].tolist()

        model = make_model()
        model.fit(X_labeled, y_labeled)

        metrics = evaluate_model(model, X_test, y_test)
        history.append({
            "round": r + 1,
            "method": "random",
            "labeled_size": len(labeled_indices),
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"]
        })

        if len(unlabeled_indices) < batch_size:
            break

        new_indices = rng.choice(unlabeled_indices, size=batch_size, replace=False)
        labeled_indices = np.concatenate([labeled_indices, new_indices])
        unlabeled_indices = np.setdiff1d(unlabeled_indices, new_indices)

    return pd.DataFrame(history)

In [ ]:
#Entropy sampling loop
def entropy_scores(probs):
    return -np.sum(probs * np.log(probs + 1e-12), axis=1)

def run_entropy_sampling(
    X_pool, y_pool, X_test, y_test,
    initial_size=60,
    batch_size=50,
    rounds=8,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    X_pool = np.array(X_pool, dtype=object)
    y_pool = np.array(y_pool)

    all_indices = np.arange(len(X_pool))

    labeled_indices = rng.choice(all_indices, size=initial_size, replace=False)
    unlabeled_indices = np.setdiff1d(all_indices, labeled_indices)

    history = []

    for r in range(rounds):
        X_labeled = X_pool[labeled_indices].tolist()
        y_labeled = y_pool[labeled_indices].tolist()

        model = make_model()
        model.fit(X_labeled, y_labeled)

        metrics = evaluate_model(model, X_test, y_test)
        history.append({
            "round": r + 1,
            "method": "entropy",
            "labeled_size": len(labeled_indices),
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"]
        })

        if len(unlabeled_indices) < batch_size:
            break

        X_unlabeled = X_pool[unlabeled_indices].tolist()
        probs = model.predict_proba(X_unlabeled)
        scores = entropy_scores(probs)

        chosen_positions = np.argsort(-scores)[:batch_size]
        new_indices = unlabeled_indices[chosen_positions]

        labeled_indices = np.concatenate([labeled_indices, new_indices])
        unlabeled_indices = np.setdiff1d(unlabeled_indices, new_indices)

    return pd.DataFrame(history)

In [ ]:
#Margin sampling loop
def margin_scores(probs):
    sorted_probs = -np.sort(-probs, axis=1)
    return sorted_probs[:, 0] - sorted_probs[:, 1]

def run_margin_sampling(
    X_pool, y_pool, X_test, y_test,
    initial_size=60,
    batch_size=50,
    rounds=8,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    X_pool = np.array(X_pool, dtype=object)
    y_pool = np.array(y_pool)

    all_indices = np.arange(len(X_pool))

    labeled_indices = rng.choice(all_indices, size=initial_size, replace=False)
    unlabeled_indices = np.setdiff1d(all_indices, labeled_indices)

    history = []

    for r in range(rounds):
        X_labeled = X_pool[labeled_indices].tolist()
        y_labeled = y_pool[labeled_indices].tolist()

        model = make_model()
        model.fit(X_labeled, y_labeled)

        metrics = evaluate_model(model, X_test, y_test)
        history.append({
            "round": r + 1,
            "method": "margin",
            "labeled_size": len(labeled_indices),
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"]
        })

        if len(unlabeled_indices) < batch_size:
            break

        X_unlabeled = X_pool[unlabeled_indices].tolist()
        probs = model.predict_proba(X_unlabeled)
        margins = margin_scores(probs)

        # smaller margin = more uncertain
        chosen_positions = np.argsort(margins)[:batch_size]
        new_indices = unlabeled_indices[chosen_positions]

        labeled_indices = np.concatenate([labeled_indices, new_indices])
        unlabeled_indices = np.setdiff1d(unlabeled_indices, new_indices)

    return pd.DataFrame(history)

In [ ]:
#Running experiments
random_results = run_random_sampling(
    X_pool, y_pool, X_test, y_test,
    initial_size=60,
    batch_size=50,
    rounds=8,
    random_state=42
)

entropy_results = run_entropy_sampling(
    X_pool, y_pool, X_test, y_test,
    initial_size=60,
    batch_size=50,
    rounds=8,
    random_state=42
)

margin_results = run_margin_sampling(
    X_pool, y_pool, X_test, y_test,
    initial_size=60,
    batch_size=50,
    rounds=8,
    random_state=42
)

all_results = pd.concat(
    [random_results, entropy_results, margin_results],
    ignore_index=True
)

all_results

In [ ]:
#Plotting macro F1 and accuracy
plt.figure(figsize=(8, 5))

for method in all_results["method"].unique():
    subset = all_results[all_results["method"] == method]
    plt.plot(
        subset["labeled_size"],
        subset["f1_macro"],
        marker="o",
        label=method
    )

plt.xlabel("Number of labeled samples")
plt.ylabel("Macro F1")
plt.title("Active Learning vs Random Sampling")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))

for method in all_results["method"].unique():
    subset = all_results[all_results["method"] == method]
    plt.plot(
        subset["labeled_size"],
        subset["accuracy"],
        marker="o",
        label=method
    )

plt.xlabel("Number of labeled samples")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Number of Labeled Samples")
plt.legend()
plt.grid(True)
plt.show()